# CoordinatorAgent - Temperature & Humidity Indicator Device

This example demonstrates how to use the `CoordinatorAgent` with the LangChain subpackage to design a temperature and humidity indicator device. The coordinator decomposes the task into three specialized subtasks handled by different worker agents:

1. **Hardware Worker** - Electronic design (sensor selection, circuit layout, power management)
2. **Mechanical Worker** - Housing design (material selection, thermal dissipation, enclosure)
3. **Firmware Worker** - Programming (sensor reading, data processing, display output)

The coordinator plans the steps, delegates to workers sequentially, and synthesizes a final design report.

In [ ]:
from typing import List, Sequence, Tuple

from a2a.types import AgentCapabilities, AgentCard, AgentSkill
from aap_core.agent import BaseAgent
from aap_core.orchestration import CoordinatorAgent
from aap_core.types import AgentMessage, BaseLLMChain
from aap_langchain.chain import ChatCausalMultiTurnsChain
from langchain_ollama import ChatOllama

## Define the Hardware Worker Chain

The hardware worker focuses on electronic design: sensor selection, circuit layout, and power management for the temperature and humidity indicator device.

In [10]:
# Ollama model setup - using qwen3:4b-thinking-2507-q4_K_M (under 30B, not embedding)
model = ChatOllama(
    model="qwen3:4b-thinking-2507-q4_K_M",
    base_url="localhost:11434",
    temperature=0.7,
    request_timeout=600,
)

# State callback for monitoring execution
def state_callback(state: str):
    print(f"[Coordinator] {state}")

In [11]:
# Hardware worker system prompt
hardware_system_prompt = """You are an expert electronic hardware engineer specializing in IoT sensor devices.
Your task is to design the electronic hardware for a temperature and humidity indicator device.

Provide a comprehensive hardware design that includes:
1. **Sensor Selection**: Recommend specific temperature and humidity sensors (e.g., DHT22, SHT31, BME280) with justification
2. **Microcontroller**: Recommend an appropriate MCU (e.g., ESP32, STM32, Arduino) based on requirements
3. **Circuit Layout**: Describe the key circuit connections and components needed
4. **Power Management**: Specify power supply options (battery, USB, solar) and power consumption considerations
5. **Display Interface**: Recommend display type (OLED, LCD, LED) and connection method

Format your response with clear sections for each area. Include specific part numbers and key specifications.
"""

hardware_chain = ChatCausalMultiTurnsChain(
    model=model,
    system_prompt=hardware_system_prompt,
    user_prompt_template="{query}",
)
hardware_chain.final_response_as_context("hardware")

## Define the Mechanical Worker Chain

The mechanical worker focuses on housing design: material selection, thermal dissipation, and enclosure dimensions.

In [12]:
# Mechanical worker system prompt
mechanical_system_prompt = """You are an expert mechanical engineer specializing in product enclosure design for electronic devices.
Your task is to design the mechanical housing for a temperature and humidity indicator device.

Provide a comprehensive mechanical design that includes:
1. **Material Selection**: Recommend materials (e.g., ABS plastic, polycarbonate, silicone) with justification for durability, cost, and environmental resistance
2. **Enclosure Dimensions**: Specify approximate dimensions and form factor (wall-mounted, portable, desktop)
3. **Thermal Management**: Design for proper heat dissipation and sensor exposure to ambient air
4. **Sensor Access**: Design openings/vents for sensors while maintaining IP rating
5. **Mounting Options**: Describe mounting methods (wall mount, stand, clip, adhesive)
6. **Aesthetic Considerations**: Color, finish, and user interface elements (buttons, LED indicators)

Format your response with clear sections for each area. Include specific material grades and manufacturing considerations (injection molding, 3D printing).
"""

mechanical_chain = ChatCausalMultiTurnsChain(
    model=model,
    system_prompt=mechanical_system_prompt,
    user_prompt_template="{query}",
)
mechanical_chain.final_response_as_context("mechanical")

## Define the Firmware Worker Chain

The firmware worker focuses on programming: sensor reading logic, data processing, and display output routines.

In [13]:
# Firmware worker system prompt
firmware_system_prompt = """You are an expert embedded firmware engineer specializing in IoT sensor devices.
Your task is to design the firmware for a temperature and humidity indicator device.

Provide a comprehensive firmware design that includes:
1. **Sensor Reading Logic**: Code/pseudocode for reading temperature and humidity sensors (I2C, SPI, or GPIO)
2. **Data Processing**: Algorithms for calibration, averaging, and threshold detection
3. **Display Output**: Code for driving the display (OLED, LCD) to show temperature and humidity readings
4. **User Interface**: Button handling, mode switching, alarm indicators
5. **Power Optimization**: Sleep modes, sampling intervals, low-power strategies
6. **Optional Features**: WiFi/Bluetooth connectivity, data logging, cloud sync

Format your response with clear sections. Include code snippets in C/C++ or MicroPython with comments.
"""

firmware_chain = ChatCausalMultiTurnsChain(
    model=model,
    system_prompt=firmware_system_prompt,
    user_prompt_template="{query}",
)
firmware_chain.final_response_as_context("firmware")

## Define Worker Agents

Create the three worker agents using the chains defined above. Each agent wraps a chain and executes it.

In [14]:
class WorkerAgent(BaseAgent):
    chain: BaseLLMChain

    def execute(self, message: AgentMessage, **kwargs) -> AgentMessage:
        self.state = "running"
        message = self.chain.invoke(message, **kwargs)
        message.execution_result = "success"
        message.origin = self.card.name
        self.state = "idle"
        return message

# Create worker agents
hardware_skill = AgentSkill(id='hardware-skill', name="hardware skill", description="Electronic hardware design for IoT devices", tags=['hardware', 'electronic'])
hardware_card = AgentCard(name="hardware_agent", description="Hardware design specialist", skills=[hardware_skill], capabilities=AgentCapabilities(), default_input_modes=['text'], default_output_modes=['text'], url="localhost", version="0.1.0")
hardware_agent = WorkerAgent(chain=hardware_chain, card=hardware_card, state_change_callback=state_callback)

mechanical_skill = AgentSkill(id='mechanical-skill', name="mechanical skill", description="Mechanical housing design for electronic devices", tags=['mechanical', 'housing'])
mechanical_card = AgentCard(name="mechanical_agent", description="Mechanical design specialist", skills=[mechanical_skill], capabilities=AgentCapabilities(), default_input_modes=['text'], default_output_modes=['text'], url="localhost", version="0.1.0")
mechanical_agent = WorkerAgent(chain=mechanical_chain, card=mechanical_card, state_change_callback=state_callback)

firmware_skill = AgentSkill(id='firmware-skill', name="firmware skill", description="Embedded firmware programming for sensor devices", tags=['firmware', 'programming'])
firmware_card = AgentCard(name="firmware_agent", description="Firmware design specialist", skills=[firmware_skill], capabilities=AgentCapabilities(), default_input_modes=['text'], default_output_modes=['text'], url="localhost", version="0.1.0")
firmware_agent = WorkerAgent(chain=firmware_chain, card=firmware_card, state_change_callback=state_callback)

workers = [hardware_agent, mechanical_agent, firmware_agent]

## Define the Planner Agent

The planner agent takes the main device query and generates a structured plan with steps, worker assignments, and dependencies.

In [15]:
# Planner agent system prompt
planner_system_prompt = """You are a project coordinator for designing a temperature and humidity indicator device.
Your job is to break down the device design task into clear, sequential steps and assign each step to the appropriate worker.

Available workers:
1. hardware_agent - Electronic hardware design (sensors, circuits, power)
2. mechanical_agent - Mechanical housing design (materials, dimensions, thermal)
3. firmware_agent - Firmware programming (sensor reading, display, logic)

For each step, specify:
- worker: which worker handles this step (one of: hardware_agent, mechanical_agent, firmware_agent)
- message: a clear instruction for the worker about what to design
- dependencies: list of indices of previous steps this step depends on (empty list if no dependencies)

Rules:
- Always start with hardware design (step 0) since it informs mechanical and firmware decisions
- Mechanical design depends on hardware (needs to know component sizes)
- Firmware design depends on hardware (needs to know which sensor/MCU is used)
- Each worker should receive the original query context plus any relevant dependencies

Output your plan as a JSON array of objects with keys: worker, message, dependencies.
Example format:
[
  {{"worker": "hardware_agent", "message": "Design the electronic hardware...", "dependencies": []}},
  {{"worker": "mechanical_agent", "message": "Design the housing...", "dependencies": [0]}},
  {{"worker": "firmware_agent", "message": "Write the firmware...", "dependencies": [0]}}
]
"""

planner_chain = ChatCausalMultiTurnsChain(
    model=model,
    system_prompt=planner_system_prompt,
    user_prompt_template="{query}",
)
planner_chain.final_response_as_context("plan")

planner_skill = AgentSkill(id='planner-skill', name="planner skill", description="Task decomposition and planning", tags=['planner'])
planner_card = AgentCard(name="planner_agent", description="Project planning coordinator", skills=[planner_skill], capabilities=AgentCapabilities(), default_input_modes=['text'], default_output_modes=['text'], url="localhost", version="0.1.0")
planner_agent = WorkerAgent(chain=planner_chain, card=planner_card, state_change_callback=state_callback)

## Implement Plan Parsing Function

This function parses the planner's JSON output into a sequence of tuples: (step_message, worker_agent, dependencies).

In [ ]:
import json
import re


def parse_plan(message: AgentMessage, workers: Sequence[BaseAgent]) -> Sequence[Tuple[AgentMessage, BaseAgent, List]]:
    """
    Parse the planner's JSON output into a sequence of (step_message, worker_agent, dependencies) tuples.
    """
    # The planner chain uses final_response_as_context("plan"), so the response is in context["plan"]
    # not in message.responses
    if message.context is None or "plan" not in message.context:
        # Fallback: try to get from responses if context is not set
        if message.responses:
            plan_text = message.responses[-1][1]
        else:
            raise ValueError("No plan found in message.context or message.responses")
    else:
        plan_text = str(message.context["plan"])

    # Try to extract JSON from the response (handle markdown code blocks)
    json_match = re.search(r'\[.*\]', plan_text, re.DOTALL)
    if not json_match:
        raise ValueError(f"Could not extract JSON plan from response: {plan_text[:200]}")

    plan = json.loads(json_match.group())

    # Map worker names to agent objects
    worker_map = {agent.card.name: agent for agent in workers}

    steps = []
    for step in plan:
        worker_name = step["worker"]
        if worker_name not in worker_map:
            raise ValueError(f"Unknown worker: {worker_name}. Available: {list(worker_map.keys())}")

        worker = worker_map[worker_name]
        dependencies = step.get("dependencies", [])

        # Create step message with context from dependencies
        step_msg = AgentMessage(query=step["message"])
        steps.append((step_msg, worker, dependencies))

    return steps

## Define Summary Agent for Final Synthesis

The summary agent aggregates results from all workers to produce a cohesive final design report.

In [17]:
# Summary agent system prompt
summary_system_prompt = """You are a senior engineering manager reviewing a temperature and humidity indicator device design.
Your job is to synthesize the hardware, mechanical, and firmware design reports into a cohesive final design document.

You will receive:
- The original device query
- Hardware design results (context_results)
- Mechanical design results (context_results)
- Firmware design results (context_results)

Create a comprehensive final design report that:
1. Summarizes the key decisions from each domain
2. Identifies any conflicts or integration challenges between domains
3. Provides a unified implementation roadmap with phases
4. Lists all recommended components with part numbers
5. Includes a bill of materials (BOM) estimate
6. Suggests next steps for prototyping

Format the report with clear headings, tables for component lists, and actionable recommendations.
"""

summary_chain = ChatCausalMultiTurnsChain(
    model=model,
    system_prompt=summary_system_prompt,
    user_prompt_template="Original query: {query}\n\nDesign results from all workers:\n{context_results}",
)
summary_chain.final_response_as_context("summary")

summary_skill = AgentSkill(id='summary-skill', name="summary skill", description="Design synthesis and report generation", tags=['summary'])
summary_card = AgentCard(name="summary_agent", description="Design synthesis coordinator", skills=[summary_skill], capabilities=AgentCapabilities(), default_input_modes=['text'], default_output_modes=['text'], url="localhost", version="0.1.0")
summary_agent = WorkerAgent(chain=summary_chain, card=summary_card, state_change_callback=state_callback)

## Instantiate and Configure CoordinatorAgent

Combine the planner, workers, summary chain, and parsing function into a single CoordinatorAgent instance.

In [18]:
# Coordinator card
coordinator_skill = AgentSkill(id='coordinator-skill', name="coordinator skill", description="Orchestrates multi-agent device design", tags=['coordinator'])
coordinator_card = AgentCard(name="coordinator_agent", description="Multi-agent device design coordinator", skills=[coordinator_skill], capabilities=AgentCapabilities(), default_input_modes=['text'], default_output_modes=['text'], url="localhost", version="0.1.0")

# Create the CoordinatorAgent
coordinator = CoordinatorAgent(
    planner_agent=planner_agent,
    parse_plan=parse_plan,
    workers=workers,
    summary_chain=summary_chain,
    summary_steps_key="context_results",
    card=coordinator_card,
    state_change_callback=state_callback,
)

## Execute Coordinator and Process Results

Run the coordinator with the device design query. The coordinator will:
1. Plan the steps (planning phase)
2. Execute each step sequentially with the appropriate worker (execution phase)
3. Synthesize a final design report (summary phase)

In [19]:
# Define the device design query
query = """Design a temperature and humidity indicator device with the following requirements:
- Display current temperature and humidity readings
- Wall-mountable for home/office use
- Battery powered with USB charging capability
- OLED display for clear readings
- Accuracy: temperature ±0.5°C, humidity ±3%
- Operating range: 0-50°C, 0-100% RH
- Low battery indicator
- Simple two-button interface (mode, reset)

Please provide a complete design covering hardware, mechanical, and firmware aspects."""

# Execute the coordinator
result = coordinator.execute(AgentMessage(query=query))

print(f"Execution result: {result.execution_result}")
print(f"Origin: {result.origin}")
print(f"Number of responses: {len(result.responses)}")

[Coordinator] coordinator_agent:planning/((planner_agent:idle)-(hardware_agent:idle)-(mechanical_agent:idle)-(firmware_agent:idle))
[Coordinator] coordinator_agent:planning/((planner_agent:running)-(hardware_agent:idle)-(mechanical_agent:idle)-(firmware_agent:idle))
[Coordinator] coordinator_agent:planning/((planner_agent:idle)-(hardware_agent:idle)-(mechanical_agent:idle)-(firmware_agent:idle))
[Coordinator] coordinator_agent:step 0: worker hardware_agent running/((planner_agent:idle)-(hardware_agent:idle)-(mechanical_agent:idle)-(firmware_agent:idle))
[Coordinator] coordinator_agent:step 0: worker hardware_agent running/((planner_agent:idle)-(hardware_agent:running)-(mechanical_agent:idle)-(firmware_agent:idle))
[Coordinator] coordinator_agent:step 0: worker hardware_agent running/((planner_agent:idle)-(hardware_agent:idle)-(mechanical_agent:idle)-(firmware_agent:idle))
[Coordinator] coordinator_agent:step 1: worker mechanical_agent running/((planner_agent:idle)-(hardware_agent:idle)

In [ ]:
# Display the final summary result
# Note: The summary chain uses final_response_as_context("summary"), so the result is in context["summary"]
if result.context and "summary" in result.context:
    summary = result.context["summary"]
    print("=== Final Design Summary ===")
    print(f"Total length: {len(summary)} characters")
    print("-" * 50)
    print(summary)
    print("-" * 50)
else:
    print("No summary found in context")
    print("Context keys:", list(result.context.keys()) if result.context else "None")

=== Final Design Summary ===
Total length: 11606 characters
--------------------------------------------------
## Comprehensive Final Design Report: Wall-Mountable Temperature and Humidity Indicator Device  

**Prepared for**: Senior Engineering Team  
**Date**: October 26, 2023  
**Target**: Home/office wall-mountable IoT device with 0–50°C accuracy  

---

### 1. Key Decisions Summary by Domain  

| **Domain**         | **Key Decision**                                                                 | **Rationale**                                                                                                                                                               |
|---------------------|--------------------------------------------------------------------------------|---------------------------------------------------------------------------------------------------------------------------------------------------------------------------|
| **Hardware**        | DHT22 sensor (Se